#프로젝트: 내가 만든 키오스크

##학습목표
####프로젝트 2번에서 Kiosk 클래스를 생성하고, 주문, 추가 주문 메서드를 적절하게 구현할 수 있다.
####프로젝트 2번의 지불, 주문표 작성 메서드를 적절하게 구현할 수 있다.
####프로젝트 2번의 지불, 주문표 작성 메서드를 적절하게 구현할 수 있다.

##평가기준
####각각의 메서드가 에러 없이, 정상적으로 동작하였다.
####각각의 메서드가 에러 없이, 정상적으로 동작하였다.
####self 인자를 잊지 않고 넣고, 변수를 알맞게 사용했다.

###메뉴와 가격 리스트를 다차원 리스트로 변경하기
###메뉴 출력 enumerate() 함수 활용하기
###datetime 모듈로 주문표에 주문 일시 추가하기
###데코레이터로 주문표 양식 출력하기

## 음료 번호 입력 단계에서 0~6 이 아닌 숫자나 문자 입력시 다시 입력하는 기능 추가
## 음료 번호 입력 단계에서 9 입력하면 주문 취소 기능 추가 (찾아주셔서 감사합니다)

In [18]:
import datetime # 모듈 불러오기 주문표에 주문 일시 추가할 예정

# [데코레이터] 영수증 양식
def receipt_decorator(func):
    def wrapper(self):
        if self.is_canceled:
            func(self)
            return
        print('⟝' + '-' * 30 + '⟞')
        for _ in range(2):
            print('|' + ' ' * 31 + '|')
        func(self)
        for _ in range(2):
            print('|' + ' ' * 31 + '|')
        print('⟝' + '-' * 30 + '⟞')
    return wrapper


class Kiosk:
    def __init__(self):
        self.menu_data = [
            ['americano', 2000],
            ['latte', 3000],
            ['mocha', 3000],
            ['yuza_tea', 2500],
            ['green_tea', 2500],
            ['choco_latte', 3000]
        ]
        self.order_menu = []
        self.order_price = []
        self.is_canceled = False # 첫 세팅 값은 False

    # 메뉴 출력
    def menu_print(self):
        for i, item in enumerate(self.menu_data):
            print(i + 1, item[0], ' : ', item[1], '원')

    # 주문 메서드 (9번 전체 취소, 0번 주문 완료!)
    def menu_select(self):
        n = -1
        while n != 0 and n != 9:

            while True:
                print()
                try:
                    n = int(input(f"음료 번호를 입력하세요 (결제 및 주문완료: 0, 전체취소: 9) : "))

                    if n == 0 or n == 9 or (1 <= n and n <= len(self.menu_data)):
                        break
                    else:
                        print(f"❌ 없는 번호입니다. 메뉴 번호나 0(결제), 9(취소)를 입력해 주세요.")

                except ValueError:
                    print("❌ 글자가 아니라 '숫자'로만 입력해 주세요.")

            # 9를 누르면 주문 취소 및 장바구니 비우기
            if n == 9:
                print("\n🚨 주문이 취소되었습니다! 장바구니를 모두 비우고 종료합니다.")
                self.order_menu = []   # 장바구니 비우기
                self.order_price = []  # 가격표 비우기
                self.is_canceled = True # 메모장에 '취소됨!' 이라고 적어두기
                break # 주문 받기 종료!

            # 0을 누르면 주문 완료 및 결제 단계로 이동
            if n == 0:
                if len(self.order_menu) == 0:
                    print("🛒 장바구니가 비어있어요! 음료를 먼저 골라주세요.")
                    n = -1 # 다시 주문을 받도록 n을 돌려놓기
                    continue
                print("\n주문이 완료되었습니다. 결제 단계로 이동합니다.")
                break

            # 음료 담기 진행
            self.price_sum = self.menu_data[n-1][1]

            t = 0
            while t != 1 and t != 2:
                try:
                    t = int(input("HOT 음료는 1을, ICE 음료는 2를 입력하세요 : "))
                    if t == 1:
                        self.temp = "HOT"
                    elif t == 2:
                        self.temp = "ICE"
                    else:
                        print("❌ 1과 2 중 하나만 골라주세요.\n")
                except ValueError:
                    print("❌ 숫자로 1 또는 2를 입력해 주세요.\n")

            menu_name = self.menu_data[n-1][0]
            print('-> 장바구니 담기 성공:', self.temp, menu_name, ':', self.price_sum, '원')

            self.order_menu.append(self.temp + ' ' + menu_name)
            self.order_price.append(self.price_sum)

    # 지불 메서드
    def pay(self):
        if self.is_canceled:
            return

        total_price = sum(self.order_price)
        print('\n지불하실 금액은', total_price, '원입니다.')

        while True:
            p = input("결제 수단을 골라주세요 (현금: 1, 카드: 2) >>> ")
            if p == '1' or p == 'cash':
                print('직원을 호출하겠습니다.\n')
                break
            elif p == '2' or p == 'card':
                print('IC칩 방향에 맞게 카드를 꽂아주세요.\n')
                break
            else:
                print('다시 결제를 시도해 주세요.')

    # 주문서 출력
    @receipt_decorator
    def table(self):
        if self.is_canceled:
            print("이용해 주셔서 감사합니다. (취소된 주문)")
            return

        now = datetime.datetime.now()
        print("주문 일시:", now.strftime("%Y-%m-%d %H:%M:%S"))
        print(" " + "-" * 30)

        for i in range(len(self.order_menu)):
            print(" ", self.order_menu[i], ' : ', self.order_price[i], '원')

        print(" " + "-" * 30)
        print('합계 금액 :', sum(self.order_price), '원')

In [19]:
# ----- [주문 하기] -----
A = Kiosk()
print("어서오세요! 1003 카페입니다. ☕\n")
A.menu_print()
A.menu_select()
A.pay()
A.table()

어서오세요! 1003 카페입니다. ☕

1 americano  :  2000 원
2 latte  :  3000 원
3 mocha  :  3000 원
4 yuza_tea  :  2500 원
5 green_tea  :  2500 원
6 choco_latte  :  3000 원

음료 번호를 입력하세요 (결제 및 주문완료: 0, 전체취소: 9) : 0
🛒 장바구니가 비어있어요! 음료를 먼저 골라주세요.

음료 번호를 입력하세요 (결제 및 주문완료: 0, 전체취소: 9) : 1
HOT 음료는 1을, ICE 음료는 2를 입력하세요 : 2
-> 장바구니 담기 성공: ICE americano : 2000 원

음료 번호를 입력하세요 (결제 및 주문완료: 0, 전체취소: 9) : 1
HOT 음료는 1을, ICE 음료는 2를 입력하세요 : 1
-> 장바구니 담기 성공: HOT americano : 2000 원

음료 번호를 입력하세요 (결제 및 주문완료: 0, 전체취소: 9) : 5
HOT 음료는 1을, ICE 음료는 2를 입력하세요 : 3
❌ 1과 2 중 하나만 골라주세요.

HOT 음료는 1을, ICE 음료는 2를 입력하세요 : 5
❌ 1과 2 중 하나만 골라주세요.

HOT 음료는 1을, ICE 음료는 2를 입력하세요 : 2
-> 장바구니 담기 성공: ICE green_tea : 2500 원

음료 번호를 입력하세요 (결제 및 주문완료: 0, 전체취소: 9) : 3
HOT 음료는 1을, ICE 음료는 2를 입력하세요 : 1
-> 장바구니 담기 성공: HOT mocha : 3000 원

음료 번호를 입력하세요 (결제 및 주문완료: 0, 전체취소: 9) : 0

주문이 완료되었습니다. 결제 단계로 이동합니다.

지불하실 금액은 9500 원입니다.
결제 수단을 골라주세요 (현금: 1, 카드: 2) >>> 1
직원을 호출하겠습니다.

⟝------------------------------⟞
|                               |
|             

#디버깅 내용
#### 코딩하면서 변수의 이름을 잘못 적었거나 무엇이 빠졌거나 하는 경우가 발생하였다.
#### 지난 퀘스트에서 try-except 구문 블록 범위 안에 할당해야했다. 각 코드 위치와 범위를 고려하였음.

#회고
#### 이 프로젝트를 진행하면서 지금까지 배워왔던 내용들을 다시 한번 상기할 수 있었다.
#### 아직 모르는 개념들이 존재하지만 지속적으로 이러한 프로젝트를 진행하다 보면 구조를 이해하고 개념도 익숙해질 것 같다.
#### 프로젝트를 진행하면서 보완하고 추가해야할 기능들이 보여 어떻게 코드를 재구성하면 좋을지 생각해보고 추가해봤다.